In [ ]:
from pathlib import Path
import re
import os

input_dir = Path("/mnt/nfs_dev/zah/data/book/JD_V2")
files = sorted([f for f in input_dir.rglob("*.md") if 'debug' not in f.parts])
print(files)

[PosixPath('/mnt/nfs_dev/zah/data/book/JD_V2/150种人造板材配方与制作 (李东光主编).pdf/150种人造板材配方与制作 (李东光主编).pdf.md'), PosixPath('/mnt/nfs_dev/zah/data/book/JD_V2/2011年西门子自动化专家会议论文集 上 (西门子（中国）有限公司编) .pdf/2011年西门子自动化专家会议论文集 上 (西门子（中国）有限公司编) .pdf.md'), PosixPath('/mnt/nfs_dev/zah/data/book/JD_V2/2011年西门子自动化专家会议论文集 下 (西门子（中国）有限公司编) .pdf/2011年西门子自动化专家会议论文集 下 (西门子（中国）有限公司编) .pdf.md'), PosixPath('/mnt/nfs_dev/zah/data/book/JD_V2/3D Integration for VLSI Systems (Chuan Seng Tan).pdf/3D Integration for VLSI Systems (Chuan Seng Tan).pdf.md'), PosixPath('/mnt/nfs_dev/zah/data/book/JD_V2/802.11无线网络权威指南 (Matthew S. Gast) .pdf/802.11无线网络权威指南 (Matthew S. Gast) .pdf.md'), PosixPath('/mnt/nfs_dev/zah/data/book/JD_V2/ADINA有限元经典实例分析 (马野，袁志丹，曹金风编著, 马野, author) .pdf/ADINA有限元经典实例分析 (马野，袁志丹，曹金风编著, 马野, author) .pdf.md'), PosixPath('/mnt/nfs_dev/zah/data/book/JD_V2/ADS信号完整性仿真与实战 (蒋修国).pdf/ADS信号完整性仿真与实战 (蒋修国).pdf.md'), PosixPath('/mnt/nfs_dev/zah/data/book/JD_V2/AI加速器架构设计与实现. (甄建勇).pdf/AI加速器架构设计与实现. (甄建勇).pdf.md'), PosixPath('

In [46]:
def has_chinese_char(text):
    """
    检测字符串中是否包含中文字符
    原理：检查字符的 Unicode 编码是否在常用汉字范围内
    """
    for char in text:
        if '\u4e00' <= char <= '\u9fff':
            return True
    return False

def is_english_file(filename):
    """
    判断文件是否为英文文件
    规则：文件名中不存在任何中文字符，即为英文文件
    """
    # 1. 获取文件名主体（去除后缀）
    # 兼容处理：如果输入是字符串路径，先转为 Path 对象
    if isinstance(filename, str):
        p = Path(filename)
    else:
        p = filename
    
    name_without_ext = p.stem
    
    # 2. 核心逻辑：如果没有中文字符，就认为是英文文件
    # 如果检测函数返回 False (无中文)，则 is_english 为 True
    return not has_chinese_char(name_without_ext)

In [47]:
def extract_toc_robust(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    start_idx = -1
    for i, line in enumerate(lines):
        clean_line = line.strip().replace(" ", "")
        if re.search(r'^#*目录$', clean_line) or re.search(r'^#*CONTENTS$', clean_line, re.I) or re.search(r'^#*目 录$', clean_line):
            start_idx = i
            break
            
    if start_idx == -1: return "False", None, None, file_path

    # --- 阶段 1: 采样前 5-8 行有效标题作为锚点集合 ---
    anchor_set = []
    anchor_clean_set = []
    scan_limit = 8 # 稍微多扫几行以确保抓到核心标题
    found_count = 0
    
    for i in range(start_idx + 1, int(len(lines)*0.2)):
        line_content = lines[i].strip().replace("#", "").strip()
        if not line_content: continue

        pure_title = re.sub(r'\s+\d+$', '', line_content).strip()
        anchor_clean_set.append(pure_title.replace(" ", "") )
        
        # if pure_title in anchor_set or pure_title in anchor_clean_set:
        #     return anchor_set
        
        anchor_set.append(pure_title)
        found_count += 1
            
        if found_count >= scan_limit or i > start_idx + 20: # 找到5个或扫描超过20行就停止
            break
    # anchor_clean_set =  [item.replace(" ", "") for item in anchor_set]
    end_start_idx = i + 1

    # --- 阶段 2: 寻找终点 ---
    end_idx = -1
    i = end_start_idx
    limit = int(len(lines) * 0.2)
    
    while i < limit:
        line_raw = lines[i].strip()
        clean_content = line_raw.replace("#", "").strip().replace(" ", "")
        
        # 1. 检查是否匹配锚点集合 (正文标题重现)
        is_anchor_match = clean_content and any(clean_content.lower() in anchor.lower() for anchor in anchor_set)
        is_anchor_clean_match = clean_content and any(clean_content.lower() in anchor.lower() for anchor in anchor_clean_set)
        is_match = any(clean_content == anchor.replace(" ", "") for anchor in anchor_set)
        has_page_num = re.search(r'\s\d+$', line_raw)
    
        if (is_anchor_match and not has_page_num) or(is_anchor_clean_match and not has_page_num) or  is_match:
            end_idx = i
            break
    
        # 2. 兜底逻辑：贪婪匹配参考文献
        # 匹配规则：去掉符号、数字、标点后，内容等于“参考文献”等
        # 使用 re.sub 去掉非中文字符和非英文字母
        simplified_content = re.sub(r'[^\u4e00-\u9fa5a-zA-Z]', '', clean_content)
        is_ref_header = re.match(r'^(参考?文献|Bibliography|References)$', simplified_content, re.IGNORECASE)
    
        if is_ref_header:
            # 初始标记为当前行
            end_idx = i + 1
            
            # 开启贪婪探测模式：向后查找 50 行
            look_ahead_idx = i + 1
            while look_ahead_idx < min(i + 51, limit):
                next_line = lines[look_ahead_idx].strip()
                next_clean = next_line.replace("#", "").strip().replace(" ", "")
                next_simplified = re.sub(r'[^\u4e00-\u9fa5a-zA-Z]', '', next_clean)
                
                # 如果在 50 行内又发现了一个独立的“参考文献”行
                if re.match(r'^(参考?文献|Bibliography|References)$', next_simplified, re.IGNORECASE):
                    end_idx = look_ahead_idx + 1
                    # 重置 i 到新发现的位置，以便外层循环能继续从这里开始下一次 50 行的探测
                    i = look_ahead_idx 
                    # 更新 look_ahead_idx 重新开始算 50 行
                    look_ahead_idx = i + 1
                    continue
                
                look_ahead_idx += 1
            
            # 探测结束，跳出外层循环
            break
        
        i += 1

    # --- 阶段 3: 返回截取结果 ---
    if end_idx != -1:
        # 如果是分行标题，end_idx 可能是标题的第二行，视情况可以向上调1行
        return lines[start_idx:end_idx], start_idx, end_idx, file_path
    else:
        # 极端情况兜底：取目录开始后的300行
        return "False", None, None, file_path
    

In [48]:
def auto_detect_hierarchy(md_content):
    # 1. 预定义可能的特征提取正则 (按优先级排序)
    feature_patterns = [
        r'第[一二三四五六七八九十\d]+篇',
        r'第[一二三四五六七八九十\d]+章',
        r'第[一二三四五六七八九十\d]+节',
        r'第[一二三四五六七八九十\d]+单元',
        r'第[一二三四五六七八九十\d]+部分',
        
        r'(?:单元|任务|项目)[一二三四五六七八九十\d]+', # 单元、任务

        r'\d+\.\d+\.\d+', # 1.1.1
        r'\d+\.\d+',      # 1.1
        r'\d+[\.．]\s*',       # 1. (带点),后面有无空格都可以，全角半角都支持
        r'^[一二三四五六七八九十]+、', # 一、
        r'^\d+\s',        # 1 (空格)
        r'^\(\d+\)',      # (1)
        r'^\([一二三四五六七八九十]+\)' # (一)
    ]

    level_queue = [] # 存储本篇文档发现的特征指纹
    output_lines = []
    
    for line in md_content:
        raw_content = line.strip()
        raw_content = raw_content.lstrip('#')
        raw_content = raw_content.strip()
        if not raw_content: continue
        
        # 移除行首旧的 # 和行尾页码
        clean_text = re.sub(r'^#+\s*', '', raw_content)
        clean_text = re.sub(r'[\.\…]{2,}', '', clean_text)
        pattern = r'\s*\(\d+\)\s*$|\s+\d+$'
        clean_text = re.sub(pattern, '', clean_text, flags=re.MULTILINE)
        
        current_fingerprint = None
        
        # 匹配特征
        for p in feature_patterns:
            match = re.search(p, clean_text)
            if match:
                # 修改部分：先标准化分隔符，再提取特征
                # 将匹配到的文本中的 '.' 或 '、' 及其后的空格统一为 '. '
                normalized_match = re.sub(r'[\.、]\s*', '. ', match.group())
                
                # 提取纯特征，忽略具体数字
                current_fingerprint = re.sub(r'\d+', '\\\\d', normalized_match)
                current_fingerprint = re.sub(r'[一二三四五六七八九十]+', 'CN', current_fingerprint)
                break
        
        if current_fingerprint:
            # 如果是新特征，加入层级队列
            if current_fingerprint not in level_queue:
                level_queue.append(current_fingerprint)
            
            # 计算当前层级 (Index 从 0 开始，所以 +1)
            depth = level_queue.index(current_fingerprint) + 1
            output_lines.append(f"{'#' * depth} {clean_text}")
        else:
            # # 如果没匹配到任何特征，视为普通文本或维持原样
            # if clean_text == "参考文献" or clean_text.lower() == "bibliography" or clean_text.lower() == "references":
            #     output_lines.append(f"{'#' * 1} {clean_text}")
            # else:
            if raw_content.startswith("#"):
                output_lines.append(raw_content)
            
    return output_lines

In [49]:
false_md = []
toc_list = []
for i in range(len(files)):
    if not (is_english_file(files[i])):
        toc_blocks, start_idx, end_idx, file_path = extract_toc_robust(files[i])
        if toc_blocks == "False":
            false_md.append(files[i])
        else:
            if toc_blocks == "False":
                false_md.append(files[i]) 
            else:
                toc_blocks = [block for block in toc_blocks if len(block) <= 50]
                print(f"------------------------{i}--------------------------")
                print(toc_blocks)
                toc_list.append((toc_blocks, start_idx, end_idx, file_path))
    else:
        pass
        # TODO：处理英文
print(false_md)


------------------------4--------------------------
['# 目录\n', '\n', '序 1\n', '\n', '前言 3\n', '\n', '第一章 无线网络导论 13\n', '\n', '为何需要无线？ 13\n', '\n', '无线网络的特色 17\n', '\n', '各式各样的网络 20\n', '\n', '第二章 802.11 网络概论 23\n', '\n', 'IEEE802网络技术族谱 24\n', '\n', '802.11相关术语及其设计 25\n', '\n', '802.11网络的运作方式 33\n', '\n', '39\n', '\n', '第三章 802.11 MAC 基础 43\n', '\n', 'MAC所面临的挑战 44\n', '\n', 'MAC 访问模式与时机 47\n', '\n', '利用DCF进行基于竞争的访问 52\n', '\n', '帧的分段与重组 55\n', '\n', '帧格式 57\n', '\n', '802.11 对上层协议的封装 65\n', '\n', '基于竞争的数据服务 66\n', '\n', '帧的处理与桥接 74\n', '\n', '# 第四章 802.11 成帧细节 78\n', '\n', '数据帧 78\n', '\n', '控制帧 87\n', '\n', '管理帧 93\n', '\n', '帧传送、关联与身份验证状态 123\n', '\n', '# 第五章 有线等效加密 127\n', '\n', 'WEP的密码学背景 128\n', '\n', 'WEP的加密操作 130\n', '\n', '关于WEP的种种问题 136\n', '\n', '动态WEP 140\n', '\n', '# 第六章 802.11x 用户身份验证 142\n', '\n', '可扩展身份验证协议 (EAP) 143\n', '\n', 'EAP认证方式（EAPMethod） 149\n', '\n', '802.1X：网络连接端口的认证 154\n', '\n', '802.1X与无线局域网 158\n', '\n', '# 第七章 802.11: RSN、TKIP 与 CCMP ..... 162\n', '\n', '临

In [50]:
processed_toc_list = []
for idx, toc in enumerate(toc_list):
    toc_content, start_idx, end_idx, file_path = toc
    processed_toc = auto_detect_hierarchy(toc_content)
    processed_toc_list.append((processed_toc, start_idx, end_idx, file_path))
# for idx, item in enumerate(processed_toc_list):
#     processwd_toc, start_idx, end_idx, file_path = item
#     rewrite_md(processwd_toc, start_idx, end_idx, file_path)

In [51]:
i = 0
processed_toc, start_idx, end_idx, file_path = processed_toc_list[i]

In [52]:
file_path

PosixPath('/mnt/nfs_dev/zah/data/book/JD_V2/802.11无线网络权威指南 (Matthew S. Gast) .pdf/802.11无线网络权威指南 (Matthew S. Gast) .pdf.md')

In [53]:
with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

In [54]:
type(lines)
print(start_idx, end_idx)

142 420


In [55]:
lines = lines[end_idx:]
lines

['# 序\n',
 '\n',
 '早在碰面之前，Matthew Gast就已经是我的心灵导师。当我发现Apple果真推出传闻已久的基于802.11b的AirPort基站，便于2000年10月开始报道无线数据网络。\n',
 '\n',
 '我曾经热衷于所谓的红外线无线网络，也曾浪费许多时间研究一些“有趣”却走入死胡同的网络技术。我原本以为802.11b不过是另一种玩具，很高兴我是错的！\n',
 '\n',
 '一路这样摸索过来，让我遇见了第一版的《802.11无线网络权威指南》。这玩意真能和宣传的一样吗？我对ISO参考模型、TCP/IP与Ethernet帧还算颇有了解，不过仍然无法将Ethernet的共享竞争（shared contention）与这种容许众声喧哗的媒介放在一块。\n',
 '\n',
 '通过文字与图表，Matthew教我一些原本我并不了解的东西。过去5年，我在《纽约时报》（New York Times）、《西雅图时报》（The Seattle Times）、PC World以及自己所开设的Wi-Fi Networking News（http://www.wifinetnews.com）网站上发表过许多文章。为了以浅显易懂的方式向一般大众解释何谓Wi-Fi，当我深入探究技术细节时这些文字图表都是我一再回顾的东西。\n',
 '\n',
 '一开始，我从《802.11无线网络权威指南》学习相关缩略术语（acronym）。通过Matthew的这本书，我后来终于超越了只知WDS代表Wireless Distribution System的程度，真正了解接入点之间如何通过允许四方协商封包传输的802.11内置机制来彼此交换数据。\n',
 '\n',
 '斗转星移，802.11家族逐渐成熟而多样化，如今本书的第一版已经稍嫌过时。不过令人惊讶的是，这些所谓的“创新”仍然根植于20世纪90年代初期至中期的技术发展。相较于2005年的“咖哩肉汤”，本书第一版的缩略术语（alphabet soup）只能用清淡来形容（译注1）。\n',
 '\n',
 "为了弥补本书第一版与无线网络发展现况之间的差距, Matthew持续在O'Reilly的Wireless DevCenter发表文章, 每一篇我都迫不及待地仔细拜读。在一场Wi-Fi Planet所举办的会议

In [56]:
processed_toc

['# 第一章 无线网络导论',
 '# 第二章 802.11 网络概论',
 '## 802.11相关术语及其设计',
 '## 802.11网络的运作方式',
 '# 第三章 802.11 MAC 基础',
 '## 802.11 对上层协议的封装',
 '# 第四章 802.11 成帧细节',
 '# 第五章 有线等效加密',
 '# 第六章 802.11x 用户身份验证',
 '## 802.1X：网络连接端口的认证',
 '## 802.1X与无线局域网',
 '# 第七章 802.11: RSN、TKIP 与 CCMP',
 '# 第八章 管理操作',
 '# 第九章PCF无竞争服务',
 '# 第十章 物理层概述',
 '### 1. 无线链路',
 '## RF传播与802.11',
 '## 802.11 的 RF 工程',
 '# 第十一章 跳频物理层',
 '# 第十二章 直接序列物理层：',
 '## DSSS与HR/DSSS(802.11b)',
 '# 第十三章 802.11a 与 802.11j: 5-GHz OFDM PHY',
 '## 802.11a所采用的OFDM',
 '# 第十四章 802.11g：增强速率物理层',
 '## 802.11g的组件',
 '# 第十五章 802.11n 前瞻：MRMO-OFDM',
 '# 第十六章 802.11 硬件',
 '## 802.11接口的一般结构',
 '# 第十七章 802.11 与 Windows',
 '# 第十八章 802.11 与 Macintosh',
 '## 在AirPort上使用802.1X',
 '# 第十九章 802.11 与 Linux',
 '# 第二十章 使用802.11接入点',
 '# 第二十一章 无线网络逻辑架构',
 '# 第二十二章 安全性架构',
 '# 第二十三章 网络规划与项目管理',
 '# 第二十四章 802.11 网络分析',
 '## 802.11网络分析项目清单',
 '# 第二十五章 802.11 性能与调整',
 '## 802.11性能评估',
 '## 802.11可调参数',
 '# 第二十六章 结论与展望']

In [57]:
type(processed_toc)

list

In [58]:
import re

def sync_toc_to_markdown(input_path, processed_toc, start_idx, end_idx):
    with open(input_path, 'r', encoding='utf-8') as f:
        original_lines = f.readlines()
        lines = original_lines[end_idx:]
    # 预处理 TOC：保留 (清洗后的文字, 原始标准格式) 的元组列表
    # 这样我们可以通过索引顺序访问
    clean_toc = []
    for t in processed_toc:
        clean_key = re.sub(r'[#\s]', '', t)
        clean_toc.append({'key': clean_key, 'full': t})

    new_lines = original_lines[:end_idx]
    toc_ptr = 0  # 当前正在寻找的 TOC 标题索引
    line_idx = 0
    num_lines = len(lines)
    num_toc = len(clean_toc)

    while line_idx < num_lines:
        line = lines[line_idx]
        stripped_line = line.strip()

        # 跳过空行，原样保留
        if not stripped_line:
            new_lines.append(line)
            line_idx += 1
            continue

        found_match = False
        
        # 核心逻辑：从当前 toc_ptr 开始往后找标题
        # 通常标题就在当前指针，但也可能文中缺失了某个标题，所以往后多看几个
        # 限制只往后看 3 个，防止跨度过大导致误匹配
        for look_ahead_toc in range(toc_ptr, min(toc_ptr + 3, num_toc)):
            target_key = clean_toc[look_ahead_toc]['key']
            
            # 滑动窗口尝试合并多行匹配当前 target_key
            temp_text = ""
            for window_size in range(0, 5): # 向后合并最多 5 行非空行
                if line_idx >= num_lines:
                    break
                if len(lines[line_idx]) > 100:

                    line_idx += 1
                if line_idx + window_size >= num_lines:
                    break
                
                curr_content = lines[line_idx + window_size].strip()
                if not curr_content and window_size > 0:
                    continue # 遇到中间空行，继续合并下一行内容
                
                temp_text += re.sub(r'[#\s]', '', curr_content)
                
                if temp_text == target_key:
                    # 匹配成功！
                    new_lines.append(clean_toc[look_ahead_toc]['full'] + '\n')
                    # 更新 TOC 指针：下次从这个标题的下一个开始找
                    toc_ptr = look_ahead_toc + 1
                    # 更新 Line 指针：跳过参与合并的所有行
                    line_idx += window_size + 1
                    found_match = True
                    break
            
            if found_match:
                break
        
        if found_match:
            continue

        # --- 如果没有匹配到 TOC ---
        if stripped_line.startswith('#'):
            # 情况：不在 TOC 中但是以 # 开头 -> 去掉 #
            new_lines.append(line.lstrip('#').lstrip())
        else:
            # 情况：普通正文 -> 原样保留
            new_lines.append(line)
        
        line_idx += 1

    return new_lines

def process_file(input_path, processed_toc, start_idx, end_idx):

    # 2. 执行转换逻辑
    result_lines = sync_toc_to_markdown(input_path, processed_toc, start_idx, end_idx)

    # 3. 构造输出路径：在原文件名后加 _2
    # splitext 分离文件名和后缀，例如 ('test', '.md')
    base_path, ext = os.path.splitext(input_path)
    output_path = f"{base_path}_2{ext}"

    # 4. 保存文件
    with open(output_path, 'w', encoding='utf-8') as f:
        f.writelines(result_lines)
    
    print(f"处理完成！\n源文件: {input_path}\n新文件: {output_path}")


# process_file(file_path, processed_toc, lines)

# 打印结果查看
# print("".join(result))

In [59]:
for idx, item in enumerate(processed_toc_list):
    processwd_toc, start_idx, end_idx, file_path = item
    process_file(file_path, processwd_toc, start_idx, end_idx)

处理完成！
源文件: /mnt/nfs_dev/zah/data/book/JD_V2/802.11无线网络权威指南 (Matthew S. Gast) .pdf/802.11无线网络权威指南 (Matthew S. Gast) .pdf.md
新文件: /mnt/nfs_dev/zah/data/book/JD_V2/802.11无线网络权威指南 (Matthew S. Gast) .pdf/802.11无线网络权威指南 (Matthew S. Gast) .pdf_2.md
处理完成！
源文件: /mnt/nfs_dev/zah/data/book/JD_V2/ADINA有限元经典实例分析 (马野，袁志丹，曹金风编著, 马野, author) .pdf/ADINA有限元经典实例分析 (马野，袁志丹，曹金风编著, 马野, author) .pdf.md
新文件: /mnt/nfs_dev/zah/data/book/JD_V2/ADINA有限元经典实例分析 (马野，袁志丹，曹金风编著, 马野, author) .pdf/ADINA有限元经典实例分析 (马野，袁志丹，曹金风编著, 马野, author) .pdf_2.md
处理完成！
源文件: /mnt/nfs_dev/zah/data/book/JD_V2/ADS信号完整性仿真与实战 (蒋修国).pdf/ADS信号完整性仿真与实战 (蒋修国).pdf.md
新文件: /mnt/nfs_dev/zah/data/book/JD_V2/ADS信号完整性仿真与实战 (蒋修国).pdf/ADS信号完整性仿真与实战 (蒋修国).pdf_2.md
处理完成！
源文件: /mnt/nfs_dev/zah/data/book/JD_V2/AI加速器架构设计与实现. (甄建勇).pdf/AI加速器架构设计与实现. (甄建勇).pdf.md
新文件: /mnt/nfs_dev/zah/data/book/JD_V2/AI加速器架构设计与实现. (甄建勇).pdf/AI加速器架构设计与实现. (甄建勇).pdf_2.md
处理完成！
源文件: /mnt/nfs_dev/zah/data/book/JD_V2/AI系统 原理与架构 (ZOMI酱, 陈仲铭, 苏统华) .pdf/AI系统 原理与架构 (ZOMI酱, 陈仲铭, 苏统华) .